# Week 3, day 4 (morning) — Worksheet 09: Loading the fact table

> *Starting from the prepared enrollment-level dataset. Looking up dimension keys
> for each fact row. Loading foreign keys that connect to dimensions. Loading
> measures and flags. Checking that each row matches the intended grain.*
> — L03, slide 38

This is **Step 6**, and everything before it has been preparation. Slide 38's own
summary of the target:

| Element | Value |
|---|---|
| Fact table | `fact_enrollment` |
| Grain | one row per student enrollment in a specific course and cohort |
| Dimension keys | `program_id, course_id, cohort_id, student_id, enrollment_date_id, promotion_id` |
| Measures | `enrollment_count, tuition_amount, discount_amount, net_tuition_amount, amount_paid_to_date` |
| Flags | `is_paid_in_full` |

Thirteen columns. Six are keys that must be *looked up* rather than copied — the
source has no `program_id` on an enrollment at all, and the surrogate keys exist
only in the dimensions built in worksheet 08.

**Question 10 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 09 — Loading the fact table. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr, tx = load("enrollment"), load("transaction")
crs, prg, coh = load("course"), load("program"), load("cohort")
stu, cat, dtype = load("students"), load("category"), load("discount_type")


def surrogate(df, key_col, new_name, unknown=True):
    """Worksheet 08's dimension builder, reduced to just the key mapping."""
    out = (df[[key_col]].drop_duplicates().sort_values(key_col)
             .reset_index(drop=True))
    out[new_name] = range(1, len(out) + 1)
    out = out.rename(columns={key_col: "source_" + key_col})
    if unknown:
        out = pd.concat([pd.DataFrame([{"source_" + key_col: -1,
                                        new_name: -1}]), out],
                        ignore_index=True)
    return out


# The six dimension key maps, as worksheet 08 built them.
K_COURSE = surrogate(crs, "course_id", "course_id_sk")
K_PROGRAM = surrogate(prg, "program_id", "program_id_sk")
K_COHORT = surrogate(coh, "cohort_id", "cohort_id_sk")
K_STUDENT = surrogate(stu, "stu_id", "student_id_sk")
K_PROMO = surrogate(dtype, "discount_type_id", "promotion_id_sk")

# Worksheet 06's row filters R5 and R6 are NOT applied to the spine here --
# question 4 needs the orphans to still be present.
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
spine = enr[~enr.stu_id.isin(test_ids)].copy()

print("enrollment spine (TEST students removed):", len(spine))
print("key maps built:", ["course", "program", "cohort", "student", "promotion"])

PART A — resolving the keys

### Question 1

Start from the spine and resolve the two easy keys: `course_id` and `cohort_id`. Print the row count after each lookup and how many rows failed to resolve.
> **NOTE:** check the row count after every lookup. A dimension key that is not unique multiplies rows here, not later.

In [ ]:
############################
## Your Code Here
############################

### Question 2

`program_id` is not on the enrollment at all — it has to come via `course`. Resolve it, and print how many enrollments got a program key.
> **NOTE:** worksheet 01 question 10 is why this needs two hops.

In [ ]:
############################
## Your Code Here
############################

### Question 3

`enrollment_date_id` is the `YYYYMMDD` integer from `dim_date`. Derive it from `enrl_date` and confirm every value falls inside the dimension's range.

In [ ]:
############################
## Your Code Here
############################

PART B — the lookup that fails

### Question 4

Resolve `student_id`. Print how many rows fail to match, then apply the `Unknown` member: unmatched rows get `student_id_sk = -1`. Print the count before and after.
> **NOTE:** worksheet 05 question 6 counted these. Confirm the number rather than trusting it.

In [ ]:
############################
## Your Code Here
############################

### Question 5

Resolve `promotion_id`, which needs the enrollment-level promotion from `transaction`. Print how many enrollments get a real promotion and how many land on the `No Promotion` member.

In [ ]:
############################
## Your Code Here
############################

PART C — assembling the table

### Question 6

Build the measures at enrollment grain, deduplicating `transaction` first (worksheet 07 question 6), and attach them to the spine with a left join. Print the row count and the null count per measure.

In [ ]:
############################
## Your Code Here
############################

### Question 7

Assemble the whole thing: all six keys, all five measures, the flag, in slide 38's column order. Print the shape, the columns, and three rows.
> **NOTE:** apply worksheet 07 question 9's guard so enrollments with no tuition are not marked paid in full.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Check the grain and the keys. Print whether `enrollment_id` is unique, the null count across the whole table, and how many rows sit on an `Unknown` member for each key.

In [ ]:
############################
## Your Code Here
############################

### Question 9

Reconcile against the source. Print the fact row count against the enrollment spine, and `SUM(amount_paid_to_date)` against the deduplicated `transaction` total for the same enrollments.
> **NOTE:** this is slide 39's row count check. It compares against something computed *outside* the fact table.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, load the fact table **without** the `Unknown` members: drop them from the key maps, resolve `student_id`, and assert no key is null. **This is supposed to fail.**

In [ ]:
############################
## Your Code Here
############################